# Module 1 · Lab 1.3 — Chain-of-Verification (CoVe): Reducing Hallucinations

**Scenario.** Retail + corporate banking (RBI guideline summarization, stock analysis)

**Learning Objectives.** By the end of this lab, you will be able to:
1. Use **Chain-of-Verification (CoVe)** to reduce factual hallucinations on document summarization.
2. Decide when to use it given a task's complexity, stakes, and latency budget.

**Tools.** LangChain v1 · `langchain-openai` · `gpt-4.1-mini`.

---


In [1]:
# 📦 Packages this notebook needs. On the training VM they are already installed,
# so the install line below stays commented out.
# Running somewhere else (Colab, your own laptop)? Uncomment it and run this cell once.
# !pip install -qU langchain langchain-core langchain-openai python-dotenv

In [2]:
import os

def load_keys(required):
    """Load API keys from Colab Secrets (on Colab) or a local .env file (elsewhere)."""
    try:
        from google.colab import userdata          # running on Google Colab
        for k in required:
            try:
                os.environ[k] = userdata.get(k)
            except Exception:
                pass                                 # not set in Colab Secrets
        source = "Colab Secrets"
    except ImportError:                              # running locally / non-Colab
        try:
            from dotenv import load_dotenv, find_dotenv
            load_dotenv(find_dotenv(usecwd=True))     # searches cwd upward for a .env
        except ImportError:
            pass
        source = ".env / environment"
    print(f"\U0001F511 Key source: {source}")
    for k in required:
        print(f"   {k:<22} {'\u2705' if os.environ.get(k) else '\u274C MISSING'}")

load_keys(["OPENAI_API_KEY"])

🔑 Key source: .env / environment
   OPENAI_API_KEY         ✅


In [11]:
import os
from IPython.display import display, Image, Markdown
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialize the model (we use a slightly higher temperature for more creative/reasoning tasks)
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.3)

# Helper function to print results neatly
def execute_prompt(prompt_template, title, **kwargs):
    """
    Executes a given prompt template with context variables and prints the output.
    """
    print(f"--- {title} ---")

    # Create the prompt using the template and input variables
    prompt = ChatPromptTemplate.from_template(prompt_template)

    # Create the chain
    chain = prompt | llm | StrOutputParser()

    # Invoke the chain with the provided context
    response = chain.invoke(kwargs)

    print("PROMPT:")
    print(prompt.format(**kwargs))
    print("\n" + "="*20 + " RESPONSE " + "="*20 + "\n")
    # print(response)
    display(Markdown(response))
    print("\n" + "-"*50 + "\n")

    return response

---

## 1. The Chain-of-Verification (CoVe) Pattern

In Lab 1.2 we saw Zero-Shot, Few-Shot, CoT, and ToT-style prompting. Now we add a **verification step** — the model double-checks its own draft against the source before producing the final answer. Empirically, CoVe substantially reduces hallucinations on document summarization and factual analysis.

**The 4-step CoVe pattern:**
1. **Draft** — generate an initial response.
2. **Plan verification questions** — what facts does the draft claim?
3. **Answer each question independently** — using only the source.
4. **Revise** — produce a final response consistent with the verified answers.


-----

## **2. Chain-of-Verification (CoVe): Minimizing Hallucinations**

### **Background**

Factual accuracy is non-negotiable in banking. **Chain-of-Verification (CoVe)** is a technique designed to make an LLM double-check its own work, significantly reducing the risk of generating plausible-sounding but incorrect information (hallucinations).

  * **How it works:** It's a deliberate, multi-step process.
    1.  **Draft Response:** The LLM generates an initial answer to the query.
    2.  **Plan Verifications:** The model reviews its own draft and generates a list of factual claims that need to be checked.
    3.  **Execute Verifications:** The model answers these verification questions *independently*, which prevents the initial draft from biasing the check.
    4.  **Final Verified Response:** The model revises the initial draft based on the outcome of the verifications, correcting any inaccuracies.
  * **Strengths:** Drastically improves the factual grounding and reliability of generated text. It's an essential technique for high-stakes content generation.
  * **Banking Use Cases:** Summarizing complex regulatory documents (like RBI circulars), generating financial reports, creating internal policy documentation, or drafting market analysis where every claim must be backed by data.

### **Use Case 2.1: Summarizing a Regulatory Guideline**

**Scenario:** A compliance officer needs a quick, accurate summary of a new (fictional) RBI guideline on digital lending.

In [37]:
rbi_guideline_text = """
RBI Circular No. 2025/11/A-34
Subject: Guidelines on Digital Lending for Non-Banking Financial Companies (NBFCs)

1. Introduction: To ensure customer protection and data security, all NBFCs engaged in digital lending must adhere to the following.
2. Loan Disbursal: All loan disbursals must be executed directly from the NBFC's bank account to the borrower's bank account. No third-party pool accounts are permitted. This rule is effective from December 1, 2025.
3. Data Privacy: Only minimal, necessary customer data may be collected. Explicit customer consent is required for any data sharing. Biometric data cannot be stored.
4. Applicability: These guidelines apply to all NBFCs. Regional Rural Banks (RRBs) are currently exempt.
"""

#### **Naive Prompt 👎**

A simple summarization request can lead to subtle but critical errors of omission or misinterpretation.

In [38]:
naive_prompt_cove_1 = "Summarize the key points of the following RBI guideline: \n\n{document}"
naive_summary = execute_prompt(
    naive_prompt_cove_1,
    "2.1 Naive Guideline Summary",
    document=rbi_guideline_text
)

--- 2.1 Naive Guideline Summary ---
PROMPT:
Human: Summarize the key points of the following RBI guideline: 


RBI Circular No. 2025/11/A-34
Subject: Guidelines on Digital Lending for Non-Banking Financial Companies (NBFCs)

1. Introduction: To ensure customer protection and data security, all NBFCs engaged in digital lending must adhere to the following.
2. Loan Disbursal: All loan disbursals must be executed directly from the NBFC's bank account to the borrower's bank account. No third-party pool accounts are permitted. This rule is effective from December 1, 2025.
3. Data Privacy: Only minimal, necessary customer data may be collected. Explicit customer consent is required for any data sharing. Biometric data cannot be stored.
4. Applicability: These guidelines apply to all NBFCs. Regional Rural Banks (RRBs) are currently exempt.


==================== RESPONSE ====================



Here is a summary of the key points from RBI Circular No. 2025/11/A-34 on Digital Lending for NBFCs:

1. **Objective**: To ensure customer protection and data security in digital lending by NBFCs.
2. **Loan Disbursal**: Loans must be disbursed directly from the NBFC’s bank account to the borrower’s bank account. Use of third-party pool accounts is prohibited, effective December 1, 2025.
3. **Data Privacy**: Only essential customer data should be collected with explicit consent for any data sharing. Storage of biometric data is not allowed.
4. **Scope**: The guidelines apply to all NBFCs, except Regional Rural Banks (RRBs), which are currently exempt.


--------------------------------------------------




**Critique:** In this run the naive summary happens to be correct — it even keeps the Regional Rural Banks (RRBs) exemption. Nothing in the prompt made the model check that, though: a summary saying the rules apply to "all financial institutions" would read just as plausibly. CoVe turns that check into an explicit step.

#### **CoVe Implementation 👍**

We simulate the CoVe process using a sequence of prompts to force self-correction.

In [39]:
# --- CoVe Step 1: Draft Initial Response ---
step1_prompt = "Draft a summary of the key points of the following RBI guideline: \n\n{document}"
draft_summary = execute_prompt(
    step1_prompt,
    "CoVe Step 1: Draft Summary",
    document=rbi_guideline_text
)


--- CoVe Step 1: Draft Summary ---
PROMPT:
Human: Draft a summary of the key points of the following RBI guideline: 


RBI Circular No. 2025/11/A-34
Subject: Guidelines on Digital Lending for Non-Banking Financial Companies (NBFCs)

1. Introduction: To ensure customer protection and data security, all NBFCs engaged in digital lending must adhere to the following.
2. Loan Disbursal: All loan disbursals must be executed directly from the NBFC's bank account to the borrower's bank account. No third-party pool accounts are permitted. This rule is effective from December 1, 2025.
3. Data Privacy: Only minimal, necessary customer data may be collected. Explicit customer consent is required for any data sharing. Biometric data cannot be stored.
4. Applicability: These guidelines apply to all NBFCs. Regional Rural Banks (RRBs) are currently exempt.


==================== RESPONSE ====================



**Summary of RBI Circular No. 2025/11/A-34: Guidelines on Digital Lending for NBFCs**

- **Objective:** To enhance customer protection and ensure data security in digital lending by NBFCs.
- **Loan Disbursal:** All loan amounts must be disbursed directly from the NBFC’s bank account to the borrower’s bank account. Use of third-party pool accounts is prohibited, effective December 1, 2025.
- **Data Privacy:** NBFCs should collect only the minimum necessary customer data. Explicit consent is mandatory for any data sharing, and storing biometric data is strictly prohibited.
- **Scope:** These guidelines apply to all NBFCs engaged in digital lending. Regional Rural Banks (RRBs) are exempt at present.


--------------------------------------------------



In [40]:

# --- CoVe Step 2: Plan Verifications ---
step2_prompt = """
Based on the following summary, generate a list of verification questions to fact-check its claims against the original document.

Summary:
{summary}
"""
verification_plan = execute_prompt(
    step2_prompt,
    "CoVe Step 2: Plan Verifications",
    summary=draft_summary
)


--- CoVe Step 2: Plan Verifications ---
PROMPT:
Human: 
Based on the following summary, generate a list of verification questions to fact-check its claims against the original document.

Summary:
**Summary of RBI Circular No. 2025/11/A-34: Guidelines on Digital Lending for NBFCs**

- **Objective:** To enhance customer protection and ensure data security in digital lending by NBFCs.
- **Loan Disbursal:** All loan amounts must be disbursed directly from the NBFC’s bank account to the borrower’s bank account. Use of third-party pool accounts is prohibited, effective December 1, 2025.
- **Data Privacy:** NBFCs should collect only the minimum necessary customer data. Explicit consent is mandatory for any data sharing, and storing biometric data is strictly prohibited.
- **Scope:** These guidelines apply to all NBFCs engaged in digital lending. Regional Rural Banks (RRBs) are exempt at present.


==================== RESPONSE ====================



Here is a list of verification questions to fact-check the summary against the original RBI Circular No. 2025/11/A-34:

1. What is the stated primary objective of RBI Circular No. 2025/11/A-34 regarding digital lending by NBFCs?
2. Does the circular explicitly require that all loan amounts be disbursed directly from the NBFC’s bank account to the borrower’s bank account?
3. Is the use of third-party pool accounts for loan disbursal prohibited, and if so, what is the effective date of this prohibition?
4. What specific guidelines does the circular provide concerning the collection of customer data by NBFCs?
5. Is explicit customer consent mandatory for any sharing of customer data according to the circular?
6. Does the circular prohibit the storage of biometric data by NBFCs?
7. To which entities do these digital lending guidelines apply? Are Regional Rural Banks (RRBs) included or exempt?
8. Are there any additional entities or types of lenders mentioned in the circular that are subject to or exempt from these guidelines?
9. Are there any other key provisions or requirements in the circular related to customer protection or data security not mentioned in the summary?
10. Does the circular specify any penalties or consequences for non-compliance with these guidelines?


--------------------------------------------------



In [41]:

# --- CoVe Step 3: Execute Verifications (Simulated) ---
# In a real system, you'd answer these questions independently. Here, we bundle them.
step3_prompt = """
Answer the following questions based *only* on the provided RBI document.

Document:
{document}

Questions:
{questions}
"""
verification_results = execute_prompt(
    step3_prompt,
    "CoVe Step 3: Execute Verifications",
    document=rbi_guideline_text,
    questions=verification_plan
)


--- CoVe Step 3: Execute Verifications ---
PROMPT:
Human: 
Answer the following questions based *only* on the provided RBI document.

Document:

RBI Circular No. 2025/11/A-34
Subject: Guidelines on Digital Lending for Non-Banking Financial Companies (NBFCs)

1. Introduction: To ensure customer protection and data security, all NBFCs engaged in digital lending must adhere to the following.
2. Loan Disbursal: All loan disbursals must be executed directly from the NBFC's bank account to the borrower's bank account. No third-party pool accounts are permitted. This rule is effective from December 1, 2025.
3. Data Privacy: Only minimal, necessary customer data may be collected. Explicit customer consent is required for any data sharing. Biometric data cannot be stored.
4. Applicability: These guidelines apply to all NBFCs. Regional Rural Banks (RRBs) are currently exempt.


Questions:
Here is a list of verification questions to fact-check the summary against the original RBI Circular No. 202

Based *only* on the provided RBI Circular No. 2025/11/A-34 document, here are the answers to the verification questions:

1. **What is the stated primary objective of RBI Circular No. 2025/11/A-34 regarding digital lending by NBFCs?**  
   The primary objective is to ensure customer protection and data security in digital lending by NBFCs.

2. **Does the circular explicitly require that all loan amounts be disbursed directly from the NBFC’s bank account to the borrower’s bank account?**  
   Yes, the circular explicitly requires that all loan disbursals must be executed directly from the NBFC's bank account to the borrower's bank account.

3. **Is the use of third-party pool accounts for loan disbursal prohibited, and if so, what is the effective date of this prohibition?**  
   Yes, the use of third-party pool accounts is prohibited. This rule is effective from December 1, 2025.

4. **What specific guidelines does the circular provide concerning the collection of customer data by NBFCs?**  
   The circular states that only minimal, necessary customer data may be collected.

5. **Is explicit customer consent mandatory for any sharing of customer data according to the circular?**  
   Yes, explicit customer consent is required for any data sharing.

6. **Does the circular prohibit the storage of biometric data by NBFCs?**  
   Yes, biometric data cannot be stored.

7. **To which entities do these digital lending guidelines apply? Are Regional Rural Banks (RRBs) included or exempt?**  
   The guidelines apply to all NBFCs. Regional Rural Banks (RRBs) are currently exempt.

8. **Are there any additional entities or types of lenders mentioned in the circular that are subject to or exempt from these guidelines?**  
   No, the circular only mentions NBFCs as subject to the guidelines and RRBs as exempt. No other entities or lender types are mentioned.

9. **Are there any other key provisions or requirements in the circular related to customer protection or data security not mentioned in the summary?**  
   Based on the provided document, no other key provisions or requirements beyond those summarized are mentioned.

10. **Does the circular specify any penalties or consequences for non-compliance with these guidelines?**  
    No, the provided document does not specify any penalties or consequences for non-compliance.


--------------------------------------------------



In [42]:

# --- CoVe Step 4: Generate Final Verified Response ---
step4_prompt = """
You have an initial draft summary and a set of verified facts.
Your task is to rewrite the summary, correcting any inaccuracies or omissions based on the verified facts.

Initial Summary:
{summary}

Verified Facts (Q&A):
{verifications}

Final Verified Summary:
"""
final_summary = execute_prompt(
    step4_prompt,
    "CoVe Step 4: Final Verified Summary",
    summary=draft_summary,
    verifications=verification_results
)

--- CoVe Step 4: Final Verified Summary ---
PROMPT:
Human: 
You have an initial draft summary and a set of verified facts.
Your task is to rewrite the summary, correcting any inaccuracies or omissions based on the verified facts.

Initial Summary:
**Summary of RBI Circular No. 2025/11/A-34: Guidelines on Digital Lending for NBFCs**

- **Objective:** To enhance customer protection and ensure data security in digital lending by NBFCs.
- **Loan Disbursal:** All loan amounts must be disbursed directly from the NBFC’s bank account to the borrower’s bank account. Use of third-party pool accounts is prohibited, effective December 1, 2025.
- **Data Privacy:** NBFCs should collect only the minimum necessary customer data. Explicit consent is mandatory for any data sharing, and storing biometric data is strictly prohibited.
- **Scope:** These guidelines apply to all NBFCs engaged in digital lending. Regional Rural Banks (RRBs) are exempt at present.

Verified Facts (Q&A):
Based *only* on the pro

**Summary of RBI Circular No. 2025/11/A-34: Guidelines on Digital Lending for NBFCs**

- **Objective:** To ensure customer protection and data security in digital lending by NBFCs.
- **Loan Disbursal:** All loan disbursals must be made directly from the NBFC’s bank account to the borrower’s bank account. The use of third-party pool accounts for loan disbursal is prohibited, effective December 1, 2025.
- **Data Privacy:** NBFCs are required to collect only the minimum necessary customer data. Explicit customer consent is mandatory for any sharing of data. The storage of biometric data by NBFCs is strictly prohibited.
- **Scope:** These guidelines apply exclusively to all NBFCs engaged in digital lending. Regional Rural Banks (RRBs) are currently exempt from these guidelines.
- **Other Provisions:** No additional entities are mentioned as subject to or exempt from these guidelines. The circular does not specify penalties or consequences for non-compliance.


--------------------------------------------------



This final version states the RRB exemption because Steps 2 and 3 asked about it and checked it against the circular — the scope is verified, not left to chance.

### **Chain-of-Verification (CoVe) Use Case 2.2: Generating Factual Stock Analysis**

**Scenario:** A junior equity analyst needs to write a short, factually precise summary of a company's quarterly performance for an internal client newsletter. The summary must not contain any speculation.

In [43]:
# Fictional quarterly data points for "Innovate Corp."
stock_data = """
- Company: Innovate Corp.
- Revenue: ₹150 Crores (an increase of 12% Year-over-Year).
- Net Profit: ₹25 Crores (an increase of 8% Year-over-Year).
- Key Highlight: Successfully launched new AI platform 'Cognito'.
- Stock Price Movement (last quarter): -5%.
- Analyst Consensus Rating: 'Hold'.
"""

#### **Naive Prompt 👎**

The model might try to be "helpful" by inventing a narrative or a cause-and-effect relationship that isn't supported by the facts provided.

In [44]:
naive_prompt_cove_2 = "Write a short paragraph analyzing Innovate Corp's latest quarterly results based on these data points: \n\n{data}"
naive_analysis = execute_prompt(
    naive_prompt_cove_2,
    "2.2 Naive Stock Analysis",
    data=stock_data
)

--- 2.2 Naive Stock Analysis ---
PROMPT:
Human: Write a short paragraph analyzing Innovate Corp's latest quarterly results based on these data points: 


- Company: Innovate Corp.
- Revenue: ₹150 Crores (an increase of 12% Year-over-Year).
- Net Profit: ₹25 Crores (an increase of 8% Year-over-Year).
- Key Highlight: Successfully launched new AI platform 'Cognito'.
- Stock Price Movement (last quarter): -5%.
- Analyst Consensus Rating: 'Hold'.


==================== RESPONSE ====================



Innovate Corp.'s latest quarterly results reflect steady growth, with revenue rising 12% year-over-year to ₹150 Crores and net profit increasing by 8% to ₹25 Crores. The successful launch of their new AI platform, 'Cognito,' marks a significant strategic milestone that could drive future innovation and revenue streams. However, despite these positive fundamentals, the stock price declined by 5% over the quarter, indicating some market hesitation. This cautious sentiment is echoed by analysts, who maintain a 'Hold' rating, suggesting that while the company shows promise, investors may be awaiting further proof of sustained growth and market impact from the new platform.


--------------------------------------------------



**Critique:** Phrases such as "indicating some market hesitation" and "investors may be awaiting further proof of sustained growth" invent reasons for the stock drop and the 'Hold' rating — causal links not present in the source data. This is speculation, not factual reporting.

#### **CoVe Implementation 👍**

The CoVe process forces the model to check its own claims against the source data, stripping out any speculative connections.

In [45]:
# --- CoVe Step 1: Draft Initial Response ---
step1_prompt = "Draft a short paragraph analyzing Innovate Corp's latest quarterly results based on these data points: \n\n{data}"
draft_analysis = execute_prompt(
    step1_prompt,
    "CoVe Step 1: Draft Analysis",
    data=stock_data
)


--- CoVe Step 1: Draft Analysis ---
PROMPT:
Human: Draft a short paragraph analyzing Innovate Corp's latest quarterly results based on these data points: 


- Company: Innovate Corp.
- Revenue: ₹150 Crores (an increase of 12% Year-over-Year).
- Net Profit: ₹25 Crores (an increase of 8% Year-over-Year).
- Key Highlight: Successfully launched new AI platform 'Cognito'.
- Stock Price Movement (last quarter): -5%.
- Analyst Consensus Rating: 'Hold'.


==================== RESPONSE ====================



Innovate Corp. demonstrated solid financial growth in its latest quarter, with revenue rising 12% year-over-year to ₹150 Crores and net profit increasing by 8% to ₹25 Crores. A significant milestone was the successful launch of their new AI platform, 'Cognito,' which positions the company well for future innovation and market expansion. However, despite these positive fundamentals, the stock price declined by 5% over the quarter, reflecting some investor caution. Correspondingly, analysts have maintained a 'Hold' rating, suggesting a wait-and-see approach as the market evaluates the impact of the new platform on long-term performance.


--------------------------------------------------



In [46]:

# --- CoVe Step 2: Plan Verifications ---
step2_prompt = """
Based on the following draft analysis, generate a list of verification questions to fact-check its claims against the original data points.

Draft Analysis:
{analysis}
"""
verification_plan_2 = execute_prompt(
    step2_prompt,
    "CoVe Step 2: Plan Verifications",
    analysis=draft_analysis
)


--- CoVe Step 2: Plan Verifications ---
PROMPT:
Human: 
Based on the following draft analysis, generate a list of verification questions to fact-check its claims against the original data points.

Draft Analysis:
Innovate Corp. demonstrated solid financial growth in its latest quarter, with revenue rising 12% year-over-year to ₹150 Crores and net profit increasing by 8% to ₹25 Crores. A significant milestone was the successful launch of their new AI platform, 'Cognito,' which positions the company well for future innovation and market expansion. However, despite these positive fundamentals, the stock price declined by 5% over the quarter, reflecting some investor caution. Correspondingly, analysts have maintained a 'Hold' rating, suggesting a wait-and-see approach as the market evaluates the impact of the new platform on long-term performance.


==================== RESPONSE ====================



Here is a list of verification questions to fact-check the draft analysis against the original data points:

1. What was Innovate Corp.'s reported revenue for the latest quarter, and how does it compare year-over-year?  
2. What was the percentage increase in revenue year-over-year for the latest quarter?  
3. What was Innovate Corp.'s net profit for the latest quarter, and how does it compare year-over-year?  
4. What was the percentage increase in net profit year-over-year for the latest quarter?  
5. Did Innovate Corp. officially launch a new AI platform named 'Cognito' during the latest quarter?  
6. Are there any statements or evidence indicating that the launch of 'Cognito' is considered a significant milestone for the company?  
7. What was the stock price movement for Innovate Corp. over the latest quarter? Specifically, did it decline by 5%?  
8. What reasons, if any, were cited for the stock price decline during the quarter?  
9. What is the current consensus analyst rating for Innovate Corp.'s stock?  
10. Have analysts provided commentary or rationale explaining the 'Hold' rating, particularly in relation to the new AI platform and its expected impact on long-term performance?


--------------------------------------------------



In [47]:

# --- CoVe Step 3: Execute Verifications ---
step3_prompt = """
Answer the following questions based *only* on the provided data points. 
Do not infer or speculate.

Data Points:
{data}

Questions:
{questions}
"""
verification_results_2 = execute_prompt(
    step3_prompt,
    "CoVe Step 3: Execute Verifications",
    data=stock_data,
    questions=verification_plan_2
)


--- CoVe Step 3: Execute Verifications ---
PROMPT:
Human: 
Answer the following questions based *only* on the provided data points. 
Do not infer or speculate.

Data Points:

- Company: Innovate Corp.
- Revenue: ₹150 Crores (an increase of 12% Year-over-Year).
- Net Profit: ₹25 Crores (an increase of 8% Year-over-Year).
- Key Highlight: Successfully launched new AI platform 'Cognito'.
- Stock Price Movement (last quarter): -5%.
- Analyst Consensus Rating: 'Hold'.


Questions:
Here is a list of verification questions to fact-check the draft analysis against the original data points:

1. What was Innovate Corp.'s reported revenue for the latest quarter, and how does it compare year-over-year?  
2. What was the percentage increase in revenue year-over-year for the latest quarter?  
3. What was Innovate Corp.'s net profit for the latest quarter, and how does it compare year-over-year?  
4. What was the percentage increase in net profit year-over-year for the latest quarter?  
5. Did Innova

1. Innovate Corp.'s reported revenue for the latest quarter was ₹150 Crores, which is an increase of 12% Year-over-Year.  
2. The percentage increase in revenue year-over-year for the latest quarter was 12%.  
3. Innovate Corp.'s net profit for the latest quarter was ₹25 Crores, which is an increase of 8% Year-over-Year.  
4. The percentage increase in net profit year-over-year for the latest quarter was 8%.  
5. Yes, Innovate Corp. officially launched a new AI platform named 'Cognito' during the latest quarter.  
6. The data points state the launch of 'Cognito' as a key highlight, but there is no explicit statement or evidence indicating it is considered a significant milestone for the company.  
7. The stock price movement for Innovate Corp. over the latest quarter was a decline of 5%.  
8. There are no reasons cited in the data points for the stock price decline during the quarter.  
9. The current consensus analyst rating for Innovate Corp.'s stock is 'Hold'.  
10. There is no commentary or rationale provided by analysts explaining the 'Hold' rating, nor any relation to the new AI platform and its expected impact on long-term performance in the provided data points.


--------------------------------------------------



In [48]:

# --- CoVe Step 4: Generate Final Verified Response ---
step4_prompt = """
You have a draft analysis and a set of verified facts.
Your task is to rewrite the analysis, ensuring every statement is directly supported by the verified facts.
Remove any speculation or causal links that are not explicitly mentioned in the data.

Draft Analysis:
{analysis}

Verified Facts (Q&A):
{verifications}

Final Verified Analysis:
"""
final_analysis = execute_prompt(
    step4_prompt,
    "CoVe Step 4: Final Verified Analysis",
    analysis=draft_analysis,
    verifications=verification_results_2
)

--- CoVe Step 4: Final Verified Analysis ---
PROMPT:
Human: 
You have a draft analysis and a set of verified facts.
Your task is to rewrite the analysis, ensuring every statement is directly supported by the verified facts.
Remove any speculation or causal links that are not explicitly mentioned in the data.

Draft Analysis:
Innovate Corp. demonstrated solid financial growth in its latest quarter, with revenue rising 12% year-over-year to ₹150 Crores and net profit increasing by 8% to ₹25 Crores. A significant milestone was the successful launch of their new AI platform, 'Cognito,' which positions the company well for future innovation and market expansion. However, despite these positive fundamentals, the stock price declined by 5% over the quarter, reflecting some investor caution. Correspondingly, analysts have maintained a 'Hold' rating, suggesting a wait-and-see approach as the market evaluates the impact of the new platform on long-term performance.

Verified Facts (Q&A):
1. Inno

Innovate Corp. reported revenue of ₹150 Crores for the latest quarter, representing a 12% increase year-over-year. Net profit for the quarter was ₹25 Crores, an 8% increase compared to the same period last year. During the quarter, the company launched a new AI platform named 'Cognito.' The stock price declined by 5% over the quarter. The current consensus analyst rating for Innovate Corp.'s stock is 'Hold.'


--------------------------------------------------



This final version is factually impeccable. It presents the different data points without inventing a narrative to connect them, which is exactly what's required for unbiased financial reporting.

---
## 3. Conclusion & Key Takeaways

### What We Covered

| Concept | Takeaway |
|---|---|
| **Chain-of-Verification (CoVe)** | Draft → plan questions → answer independently → revise — substantial hallucination reduction |
| **Cost / latency trade-off** | CoVe is ~3-4× total tokens. Worth it when accuracy matters more than speed |

**Next Lab:** Lab 1.4 — Prompt Caching Across Providers (OpenAI · Anthropic · Gemini)


## 4. Stretch Exercise (Optional)

1. Build a 10-document factual benchmark and measure CoVe's hallucination-reduction empirically (LLM-as-Judge for groundedness).
2. Add a 'CoVe-on-CoT' compound: verify each step of a CoT reasoning chain, not just the final answer.
3. Wire the CoVe loop into a LangGraph state machine — a preview of the LangGraph workflow labs, Labs 2.3–2.6.
